In [1]:
from troncamento_datasets import BaseDataset, SegmentPairDataset
from model import MisalignmentDetector
import torch, torch.nn as nn
import pandas as pd
import os
import tqdm
import utils

In [2]:
device = "mps"

model = MisalignmentDetector().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss(reduction="none")

ckpt = "model_ckpt/train.pt"
if os.path.isfile(ckpt):
    model.load_state_dict(torch.load(ckpt))
    print(f"Loaded trained model from {ckpt}")

Loaded trained model from model_ckpt/train.pt


In [10]:
import glob
saved_embed_fs = glob.glob("saved_embeds/*.npy")

df = pd.read_csv("troncamento_data.csv")
df = df[df["sent_it_prob"]>0.99].reset_index(drop=True)

selected_ids = [int(os.path.basename(f).split("_")[0]) for f in saved_embed_fs]
df = df[df["id"].isin(selected_ids)].reset_index(drop=True)

In [11]:
basedataset = BaseDataset(df, dataset_type="all", return_player=False)

In [14]:
df["predicted_label"] = df.index.map(basedataset.predict_label)

In [16]:
df.to_csv("troncamento_predictions.csv", index=False)